# M37 lab solution: a research team with edge cases

**Brief.** Build a supervisor with a RAG researcher, a web researcher and a writer that cites sources, then test it on edge cases (no results, conflicting sources, tool failure).

The team answers Larkspur Cloud questions:

- **RAG researcher.** TF-IDF over the handbook and the lab's `conflicting.md` forum post, with a relevance floor: when no chunk clears it, the node returns no findings and makes no model call (the 04b mechanism).
- **Web researcher.** Tavily search (`langchain_tavily.TavilySearch`). A failing call is caught and written to state as an error, never raised out of the graph (the 02c and 04a mechanism).
- **Supervisor.** Reads what has been found so far and picks the next worker or the writer. The choice is a typed field, and a pure router function enforces `MAX_SUPERVISOR_TURNS`.
- **Writer.** Answers from the findings only, cites each claim by source id, lists conflicts between sources, and says which source is the policy. Its citations are checked in code against the sources the researchers returned.

Larkspur Cloud is fictional, so the web researcher will not find its handbook. It is here to show the role and to carry the tool-failure case.

## Setup

In [ ]:
import json
import operator
import sys
from pathlib import Path
from typing import Annotated, Callable, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_tavily import TavilySearch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from typing_extensions import TypedDict

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import load_handbook, load_jsonl  # noqa: E402

DATA = Path("../data")
llm = get_experientiallabs_llm()
MODEL = llm.model_name
golden = {g["id"]: g for g in load_jsonl(DATA / "larkspur" / "golden.jsonl")}
print("model:", MODEL)

## The RAG researcher's corpus

Every chunk keeps its source id: `handbook/<file>` for the official handbook, `forum/conflicting.md` for the forum post.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
corpus = [
    {"source": f"handbook/{page['source']}", "kind": "handbook", "text": chunk}
    for page in load_handbook(DATA / "larkspur" / "handbook")
    for chunk in splitter.split_text(page["text"])
]
forum = (DATA / "conflicting.md").read_text()
corpus += [{"source": "forum/conflicting.md", "kind": "forum", "text": c} for c in splitter.split_text(forum)]

vectorizer = TfidfVectorizer(stop_words="english")
matrix = vectorizer.fit_transform([c["text"] for c in corpus])
RELEVANCE_FLOOR = 0.1  # TF-IDF cosine; below it a chunk is not evidence. Print scores for your own questions before changing it.


def rag_search(query: str, k: int = 4) -> list[dict]:
    scores = (matrix @ vectorizer.transform([query]).T).toarray().ravel()
    order = sorted(range(len(corpus)), key=lambda i: -scores[i])[:k]
    return [{**corpus[i], "score": round(float(scores[i]), 3)} for i in order if scores[i] >= RELEVANCE_FLOOR]


print(len(corpus), "chunks")
for hit in rag_search(golden["g01"]["question"]):
    print(hit["score"], hit["source"], hit["text"][:80])

## The web researcher's tool

`web_search` is a plain function so an edge case can swap it: `failing_search` raises, `empty_search` returns nothing.

In [ ]:
tavily = TavilySearch(max_results=3)


def tavily_search(query: str) -> list[dict]:
    out = tavily.invoke({"query": query})
    return [{"source": r["url"], "kind": "web", "text": r.get("content", "")} for r in out.get("results", [])]


def failing_search(query: str) -> list[dict]:
    raise RuntimeError("web search provider timed out")


def empty_search(query: str) -> list[dict]:
    return []

## State, workers, supervisor and writer

In [ ]:
MAX_SUPERVISOR_TURNS = 4


class TeamState(TypedDict):
    question: str
    findings: Annotated[list[dict], operator.add]
    errors: Annotated[list[str], operator.add]
    done: Annotated[list[str], operator.add]
    turns: int
    next: str
    report: dict


class Decision(BaseModel):
    next: Literal["rag_researcher", "web_researcher", "writer"] = Field(
        description="Who works next. Use writer once the findings answer the question or no researcher can add more."
    )


class Report(BaseModel):
    answer: str = Field(description="The answer, with a [source id] after each claim. If nothing answers it, say so.")
    citations: list[str] = Field(description="Every source id cited in the answer.")
    conflicts: list[str] = Field(description="Each disagreement between sources, naming both source ids. Empty if none.")
    policy_source: str = Field(description="When sources conflict, the source id that states official policy; else empty.")


def make_team(web_search: Callable[[str], list[dict]]):
    decide = llm.with_structured_output(Decision)
    write = llm.with_structured_output(Report)

    def rag_researcher(state: TeamState) -> dict:
        hits = rag_search(state["question"])  # no model call when nothing clears the floor
        return {"findings": hits, "done": ["rag_researcher"]}

    def web_researcher(state: TeamState) -> dict:
        try:
            return {"findings": web_search(state["question"]), "done": ["web_researcher"]}
        except Exception as exc:  # noqa: BLE001
            return {"errors": [f"web_researcher: {type(exc).__name__}: {exc}"], "done": ["web_researcher"]}

    def supervisor(state: TeamState) -> dict:
        summary = "\n".join(f"- [{f['source']}] {f['text'][:200]}" for f in state["findings"]) or "(no findings)"
        d = decide.invoke([
            SystemMessage("You lead a research team: rag_researcher searches the Larkspur Cloud handbook and forum, "
                          "web_researcher searches the web, writer writes the answer. Pick who works next."),
            HumanMessage(f"Question: {state['question']}\nAlready ran: {state['done']}\nErrors: {state['errors']}\n"
                         f"Findings:\n{summary}"),
        ])
        return {"next": d.next, "turns": state["turns"] + 1}

    def route(state: TeamState) -> str:
        # The caps live here, in code: no worker runs twice, and the writer runs once the turn budget is spent.
        if state["turns"] >= MAX_SUPERVISOR_TURNS or state["next"] == "writer" or state["next"] in state["done"]:
            return "writer"
        return state["next"]

    def writer(state: TeamState) -> dict:
        sources = "\n".join(f"[{f['source']}] ({f['kind']}) {f['text']}" for f in state["findings"]) or "(none)"
        r = write.invoke([
            SystemMessage("Write a short answer for a Larkspur Cloud customer from the findings only. Cite every claim "
                          "with its [source id]. Handbook sources state official policy; forum and web sources do not. "
                          "If sources disagree, give both and say which is the policy. If the findings do not answer "
                          "the question, say that the team found no answer. Mention any tool errors."),
            HumanMessage(f"Question: {state['question']}\n\nFindings:\n{sources}\n\nTool errors: {state['errors'] or 'none'}"),
        ])
        known = {f["source"] for f in state["findings"]}
        report = r.model_dump()
        report["unverified_citations"] = [c for c in r.citations if c not in known]  # checked in code, not by the model
        return {"report": report}

    g = StateGraph(TeamState)
    for name, fn in (("supervisor", supervisor), ("rag_researcher", rag_researcher),
                     ("web_researcher", web_researcher), ("writer", writer)):
        g.add_node(name, fn)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", route, ["rag_researcher", "web_researcher", "writer"])
    g.add_edge("rag_researcher", "supervisor")
    g.add_edge("web_researcher", "supervisor")
    g.add_edge("writer", END)
    return g.compile()


def run(question: str, web_search: Callable[[str], list[dict]] = tavily_search) -> dict:
    team = make_team(web_search)
    state = {"question": question, "findings": [], "errors": [], "done": [], "turns": 0, "next": "", "report": {}}
    path = []
    final = None
    for mode, chunk in team.stream(state, {"recursion_limit": 3 * MAX_SUPERVISOR_TURNS + 4}, stream_mode=["updates", "values"]):
        if mode == "updates":
            path += list(chunk)
        else:
            final = chunk
    final["path"] = path
    return final


def show(name: str, out: dict) -> None:
    r = out["report"]
    print(f"=== {name}\npath: {' -> '.join(out['path'])}")
    print("sources found:", sorted({f['source'] for f in out['findings']}) or "none", "| errors:", out["errors"] or "none")
    print("answer:", r["answer"])
    print("citations:", r["citations"], "| unverified:", r["unverified_citations"])
    print("conflicts:", r["conflicts"] or "none", "| policy source:", r["policy_source"] or "-")
    print()

## Baseline: a question the handbook answers

In [ ]:
baseline = run(golden["g06"]["question"])
show("baseline: " + golden["g06"]["question"], baseline)
print("reference:", golden["g06"]["reference_answer"])

## Edge case 1: no results

`g29` asks what discount non-profit organisations get; the handbook does not say, and none of its words match a chunk, so the RAG researcher returns nothing above the floor, and the web tool is swapped for `empty_search` so the case does not depend on what the web returns for a fictional company. The writer should say that the team found no answer, with no citations.

In [ ]:
no_results = run(golden["g29"]["question"], web_search=empty_search)
show("no results: " + golden["g29"]["question"], no_results)
assert not no_results["findings"], "expected no findings"

## Edge case 2: conflicting sources

The handbook says refunds take 5 to 10 business days; the forum post says 2 to 3. Both should reach the writer with their source ids, and the writer should name the conflict and the handbook as the policy.

In [ ]:
conflict = run("How long does a refund take to appear on my card statement?")
show("conflicting sources", conflict)
found = {f["source"] for f in conflict["findings"]}
print("handbook and forum both retrieved:", {"handbook/02_refunds.md", "forum/conflicting.md"} <= found)

## Edge case 3: a tool that raises

`failing_search` raises on every call. The web researcher records the error in state, the supervisor sees it, and the graph still ends at the writer.

In [ ]:
tool_failure = run(golden["g12"]["question"], web_search=failing_search)
show("tool failure: " + golden["g12"]["question"], tool_failure)
assert tool_failure["path"][-1] == "writer", "the graph must end at the writer"

## Results

In [ ]:
import pandas as pd

rows = []
for name, out, expected in (
    ("baseline (g06)", baseline, "cited handbook answer"),
    ("no results (g29)", no_results, "says no answer, no citations"),
    ("conflicting sources", conflict, "both sources cited, handbook named as policy"),
    ("tool failure (g12)", tool_failure, "error recorded, answer from handbook"),
):
    r = out["report"]
    rows.append({"case": name, "expected": expected, "path": " > ".join(out["path"]),
                 "findings": len(out["findings"]), "errors": len(out["errors"]),
                 "citations": len(r["citations"]), "unverified": len(r["unverified_citations"]),
                 "conflicts": len(r["conflicts"]), "policy_source": r["policy_source"] or "-"})
print("model:", MODEL)
print(pd.DataFrame(rows).to_markdown(index=False))

## What this shows

- Each edge case is handled by a mechanism in code: the relevance floor (no model call, no findings), source ids carried from retrieval to the writer (conflicts stay attributable), and a `try` in the worker plus a capped router (a failing tool cannot stop the run).
- Checking citations against the returned sources is cheap and catches an invented source id, which a prompt instruction alone cannot.
- Read the `path` column: a supervisor that keeps choosing workers already in `done` is stopped by `route()`, so the cost of a run is bounded by `MAX_SUPERVISOR_TURNS`.